<a href="https://colab.research.google.com/github/hamzafareed123/code-review-agent/blob/fetch-pr-node/code_review_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB-TOKEN")
GROQ_API_KEY=  userdata.get("GROQ_API_KEY")


In [ ]:
!pip install PyGithub langgraph langchain langchain-core langchain-community langchain-groq

In [ ]:
from github import Github, Auth
from langgraph.graph import START,END,StateGraph
from typing import TypedDict,Literal
from pydantic import BaseModel,Field
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
#==================================
#  LLM setup
#==================================

llm = ChatGroq(model="openai/gpt-oss-120b", api_key=GROQ_API_KEY,  temperature=0.2, max_tokens=1024)


In [ ]:
class BugFinding(BaseModel):
    file_name: str
    line_number: int
    severity: Literal["Critical", "Medium", "Low"]
    description: str
    recommendation: str

In [ ]:
#==================================
#  Graph STATE
#==================================

class PRState(TypedDict):
  repo_name:str
  pr_title:str
  pr_body:str
  pr_number:int
  pr_state:Literal["open","closed","merged"]
  files:list[dict]

  bug_findings:list[BugFinding]

In [ ]:
#==================================
#  Fetch Pull Request NODE
#==================================

def fetch_pr_node(state:PRState):
  auth = Auth.Token(GITHUB_TOKEN)
  g= Github(auth=auth)
  repo = g.get_repo(state["repo_name"])
  pr= repo.get_pull(state['pr_number'])

  files = pr.get_files()

  for f in files:
    state["files"].append({
        "filename": f.filename,
        "additions": f.additions,
        "deletions": f.deletions,
        "changes": f.changes,
        "patch": f.patch  })

  pr_body_content = pr.body if pr.body else "No Description Provided"
  return {"pr_title":pr.title,"pr_body":pr_body_content,"pr_number":pr.number,"pr_state":pr.state}

In [ ]:
#==================================
#  Bug Finding NODE
#==================================

bug_find_prompt = ChatPromptTemplate.from_template(
    "You are a senior code reviewer performing automated static analysis.\n\n"
    "Analyze ONLY the diff below for genuine bugs, logic errors, null/undefined risks, "
    "off-by-one errors, race conditions, and incorrect API usage. "
    "Do NOT comment on code style, formatting, naming, or performance.\n\n"
    "Respond with ONLY a single structured tool call matching the required schema. "
    "Do not write any prose, explanation, markdown, tables, or commentary before or after the tool call. "
    "If you find no bugs, still return one finding with severity 'Low' and description 'No significant bugs found'.\n\n"
    "Required fields:\n"
    "- file_name: the exact filename given below\n"
    "- line_number: the closest line number in the diff where the issue occurs (use 0 if not applicable)\n"
    "- severity: exactly one of 'Critical', 'Medium', or 'Low'\n"
    "- description: one concise sentence describing the bug\n"
    "- recommendation: one concise sentence describing the fix\n\n"
    "Filename: {filename}\n"
    "Diff:\n{patch}"
)

bug_find_llm = llm.with_structured_output(BugFinding)
bug_find_chain = bug_find_prompt | bug_find_llm

def bug_finding_node(state: PRState):
    bug_findings = []
    for f in state["files"]:
        if f["patch"] is None:
            continue
        result = bug_find_chain.invoke({"filename": f["filename"], "patch": f["patch"]})
        bug_findings.append(result)
    return {"bug_findings": bug_findings}

In [ ]:
#==================================
#  Graph Build STATE
#==================================

graph = StateGraph(PRState)

graph.add_node("fetch_pr_node",fetch_pr_node)
graph.add_node("bug_finding_node",bug_finding_node)

graph.add_edge(START,"fetch_pr_node")
graph.add_edge("fetch_pr_node","bug_finding_node")
graph.add_edge("bug_finding_node",END)

workflow = graph.compile()

In [ ]:
workflow

In [ ]:
initial_state = {"repo_name":"hamzafareed123/async-board","pr_title":"","pr_body":"","pr_number":49,"pr_state":"open","files":[]}

result=workflow.invoke(initial_state)


In [ ]:
print(result)

for i, b in enumerate(result['bug_findings']):
  print(f"\n========== Bug No {i+1}\n")
  print(b.file_name)
  print(b.line_number)
  print(b.severity)
  print(b.description)
  print(b.recommendation)